# Homunculus Syndicate — 윤수르 V4 파인튜닝 (Unsloth + TRL)

**v3 대비 바뀐 것은 데이터뿐이다.** 하이퍼파라미터(LoRA r=16/α=32, lr 2e-4, 1 epoch, bf16, seq 2048)는 v3와 동일하게 고정해 "데이터 재설계만의 효과"를 분리한다. 실험 노트: `docs/experiments/yunsur_v4/`.

- **데이터:** `train_data_v4.jsonl` — 1,099건, `{slot, system, instruction, output}`. RAG 400 / 잡담 350 / 계획 249 / 코딩 100.
- **system은 레코드마다 다르다** — 운영 봇이 그 슬롯에 실제로 주입하는 프롬프트. v3의 `SYSTEM_PERSONA` 단일 덮어쓰기는 폐기.
- **기본 베이스:** `Qwen/Qwen3.5-9B` — **`transformers>=5.2.0` 필수** (`qwen3_5` 아키텍처).
- **GGUF:** 실패 시 LoRA만 저장하고 맥에서 변환해도 됨. 맥미니에서 `ollama create yunsur_v4 -f Modelfile.v4`.


## 0. 설치 (RunPod / Linux, 셀당 1회)

**중요:** `unsloth_zoo`는 `torch._inductor.config`가 있는 **PyTorch 2.x(권장 2.5+)** 를 요구합니다. 구버전 torch만 깔린 이미지에서 Unsloth만 올리면  
`AttributeError: module 'torch._inductor' has no attribute 'config'` 가 납니다.

**또 다른 흔한 오류:** `torch.utils._pytree`에 `register_constant` 없음 / `torchao` 스택  
→ **최신 `torchao` + 구형 `torch`** 조합입니다. 아래 셀에서 `torchao`를 빼 주면(LoRA SFT에는 보통 불필요) `transformers` import가 통과하는 경우가 많습니다.  
근본 해결은 **PyTorch를 최신 2.5.x+로 올리는 것**(아래 cu124 휠).

**또:** `cannot import name 'device_synchronize' from 'unsloth_zoo.device_type'`  
→ **`unsloth`(git)와 `unsloth_zoo`(PyPI 구버전)** 가 짝이 안 맞을 때입니다. 아래 셀에서 **zoo를 git main으로 맞춘 뒤** 커널 재시작하세요.

1. **아래 셀에서 torch를 먼저** 올린 뒤 Unsloth 설치  
2. pip로 패키지를 바꾼 직후에는 **반드시 커널 재시작** → 그다음 **검증 셀** 실행  

CUDA 휠 URL은 Pod에 맞게 바꾸세요: `cu124` / `cu121` / `cu118` ([PyTorch 시작하기](https://pytorch.org/get-started/locally/)).

**Qwen3.5 사용 시:** `KeyError: 'qwen3_5'` / `transformers … does not support Qwen3.5` → **`pip install -U "transformers>=5.2.0"`** 후 커널 재시작. PyPI에 없으면 `pip install git+https://github.com/huggingface/transformers.git`.

In [ ]:
# ① PyTorch 먼저 (Unsloth Zoo가 torch._inductor.config 필요 — 구버전이면 import 단계에서 터짐)
%pip install -U pip
%pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# CUDA 11.8 이미지면 위 한 줄을 cu118 로 바꿀 것: .../whl/cu118

# ② transformers 경로에서 torchao가 깔려 있으면 구형 torch와 충돌할 수 있음 (register_constant 오류)
#    LoRA 학습만 할 때는 torchao 제거해도 무방한 경우가 많음
%pip uninstall -y torchao

# ③ Unsloth (git) — 직후에 unsloth_zoo를 같은 출처로 맞춰야 함 (PyPI 구버전 zoo면 device_synchronize ImportError)
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# 실패 시: %pip install unsloth

# ③-b unsloth_zoo를 git main으로 강제 동기화 (unsloth와 API 짝 맞추기)
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps "git+https://github.com/unslothai/unsloth-zoo.git"

# ④ 나머지 학습 스택 (여기서 unsloth_zoo를 또 올리지 말 것 — 구 PyPI가 깃 unsloth를 깨뜨림)
%pip install -U datasets accelerate peft trl bitsandbytes hf_transfer

# ④-b Qwen3.5 (model_type qwen3_5) — transformers 4.x에는 없음. Unsloth: 최소 5.2.0
%pip install -U "transformers>=5.2.0"
# 위가 의존성 충돌이면: %pip install git+https://github.com/huggingface/transformers.git

# ⑤ 다른 패키지가 torchao를 다시 끌어왔을 수 있음 → 한 번 더 제거
%pip uninstall -y torchao

### 커널 재시작 (pip 직후 필수)

위 셀에서 `torch`를 새로 깔았다면 **지금 Jupyter / RunPod 커널을 재시작**한 뒤, **아래 검증 셀 → 그다음 셀부터** 순서로 다시 실행하세요. (재시작 없이 하면 예전 torch가 메모리에 남아 동일 오류가 납니다.)

In [ ]:
import torch

# unsloth_zoo → common.py 가 inspect.getsource(torch._inductor.config) 를 쓰므로 config 모듈 필수
_ind = getattr(torch, "_inductor", None)
_cfg = getattr(_ind, "config", None) if _ind is not None else None
if _cfg is None:
    raise RuntimeError(
        "torch._inductor.config 를 찾을 수 없습니다.\n"
        "→ PyTorch가 너무 오래됐거나, pip로 torch를 올린 뒤 커널을 재시작하지 않은 경우가 많습니다.\n"
        "조치: 설치 셀에서 torch를 cu에 맞게 upgrade → 커널 재시작 → 이 셀부터 재실행."
    )

# torchao 최신판이 구형 torch와 섞이면 register_constant 관련 AttributeError (설치 셀에서 torchao 제거 권장)
_pt = getattr(torch.utils, "_pytree", None)
if _pt is not None and not hasattr(_pt, "register_constant"):
    print(
        "[경고] torch.utils._pytree.register_constant 없음 — "
        "설치 셀의 `%pip uninstall -y torchao` 실행 후 커널 재시작, 또는 PyTorch를 2.5+로 올리세요."
    )

import transformers

print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| inductor OK")
print("transformers:", transformers.__version__, "| (Qwen3.5 / qwen3_5 는 >= 5.2.0 필요)")

In [ ]:
# (선택) 대용량 HF 다운로드 가속
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

## 1. 설정 · 경로

In [ ]:
from pathlib import Path

# ── 데이터 파일: 노트북 실행 위치 기준 (RunPod에서는 작업 폴더에 jsonl 업로드 후 cwd 맞추기)
DATA_FILE = Path.cwd() / "train_data_v4.jsonl"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"데이터 없음: {DATA_FILE.resolve()}\n"
        "jsonl을 이 노트북과 같은 디렉터리에 두거나, DATA_FILE 경로를 수정하세요."
    )

OUTPUT_DIR = Path("outputs") / "yunsur_v4_lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Qwen3.5 공식 채팅 가중치 (Instruct 전용 리포명 아님)
MODEL_NAME = "Qwen/Qwen3.5-9B"
# 텍스트 SFT만 안정적으로 검증하려면 대안:
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

max_seq_length = 2048
load_in_4bit = False

## 2. 모델 · 토크나이저 로드

In [ ]:
import torch
from unsloth import FastLanguageModel

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("🧠 베이스 모델 로드…", MODEL_NAME)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. LoRA

In [ ]:
print("⚙️ LoRA 장착…")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. ChatML 템플릿 + ShareGPT 변환

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
)

# v4: system 은 레코드 필드를 그대로 쓴다 (슬롯별 운영 프롬프트). 단일 페르소나 덮어쓰기 금지.
REQUIRED_KEYS = ("slot", "system", "instruction", "output")


def convert_to_sharegpt(examples):
    convos = []
    for system, instruction, output in zip(examples["system"], examples["instruction"], examples["output"]):
        assert system and system.strip(), "system 이 비어 있는 레코드 — v4 데이터 규약 위반"
        convos.append(
            [
                {"from": "system", "value": system},
                {"from": "human", "value": instruction},
                {"from": "gpt", "value": output},
            ]
        )
    return {"conversations": convos}


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=str(DATA_FILE.resolve()),
    split="train",
)
dataset = dataset.map(convert_to_sharegpt, batched=True)
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched=True)
print("샘플 수:", len(dataset))

# 03_train_config.md 에 기록할 값 — 슬롯 분포 / system 종류 수 / 토큰 길이
from collections import Counter
raw_rows = load_dataset("json", data_files=str(DATA_FILE.resolve()), split="train")
print("슬롯 분포:", dict(Counter(raw_rows["slot"])))
print("system 종류:", len(set(raw_rows["system"])))
_tok = getattr(tokenizer, "tokenizer", tokenizer)  # Qwen3.5 는 Processor — 위치 인자는 images 로 해석되므로 text= 로
lens = [len(_tok(text=t)["input_ids"]) for t in dataset["text"]]
lens.sort()
print(f"토큰 길이: 중앙값 {lens[len(lens)//2]}, p95 {lens[int(len(lens)*0.95)]}, max {lens[-1]} (max_seq_length={max_seq_length})")
print("잘리는 샘플(max_seq 초과):", sum(1 for l in lens if l > max_seq_length))

## 5. 마스킹용 문자열 검증 (필수)

`train_on_responses_only`에 넣을 문자열이 **실제 `text`에 그대로** 있어야 합니다. 없으면 아래 `INSTRUCTION_MARK` / `RESPONSE_MARK`를 `repr` 출력에 맞게 수정하세요.

In [ ]:
sample = dataset[0]["text"]
print(sample[:2500])
print("--- repr (앞부분) ---")
print(repr(sample[:1200]))

In [ ]:
# ChatML + Qwen 계열에서 흔한 구분자 (위 셀 출력과 다르면 여기만 고침)
INSTRUCTION_MARK = "<|im_start|>user\n"
RESPONSE_MARK = "<|im_start|>assistant\n"

assert RESPONSE_MARK in sample, (
    f"템플릿에 '{RESPONSE_MARK}' 없음. tokenizer/chat_template 또는 RESPONSE_MARK 수정."
)
print("마스킹 delimiter 검증 OK")

## 6. Trainer

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=str(OUTPUT_DIR),
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_MARK,
    response_part=RESPONSE_MARK,
)

In [ ]:
print("🚀 학습 시작")
trainer_stats = trainer.train()
trainer_stats

# 03_train_config.md 용: loss 로그를 파일로 남긴다 (logging_steps=10 기준)
import json as _json
hist = [h for h in trainer.state.log_history if "loss" in h]
Path("yunsur_v4_train_log.json").write_text(_json.dumps(trainer.state.log_history, ensure_ascii=False, indent=2))
if hist:
    print(f"loss 시작 {hist[0]['loss']:.4f} → 끝 {hist[-1]['loss']:.4f} ({len(hist)}회 기록, 총 step {trainer.state.global_step})")
print("학습 로그 저장: yunsur_v4_train_log.json — 노트북과 함께 맥으로 가져올 것")


## 7. 저장 (LoRA + 토크나이저)

맥/Ollama 등은 이 폴더를 받아 병합·변환하면 됩니다.

In [ ]:
FINAL_DIR = Path("yunsur_v4_model")
FINAL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print("저장 완료:", FINAL_DIR.resolve())

## 8. (선택) GGUF `q8_0` — 환경에 따라 실패할 수 있음

In [ ]:
try:
    model.save_pretrained_gguf(
        str(FINAL_DIR),
        tokenizer,
        quantization_method="q8_0",
    )
    print("GGUF 저장 시도 완료 — 폴더 내 .gguf 확인")
except Exception as e:
    print("GGUF 스킵:", e)
    print("LoRA/가중치 폴더만으로도 맥에서 llama.cpp 등으로 변환 가능합니다.")